<a href="https://colab.research.google.com/github/pgordin/GraphsSN_2026_1/blob/main/Graphs4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Graphs & Social Networks &mdash; lab material
This notebook contains a definition of a python class *Graph*, containing functions described during labs.

Package import.

In [56]:
import numpy as np
from random import random, seed
from copy import deepcopy

## Simple functions from the first classes
This is left just as an example (to compare the behaviour).

In [2]:
def print_matrix(vertices, matrix):
  """
  Prints a graph given by the matrix.
  """
  n = len(matrix)   # the order of a graph
  # take the proper names of vertices
  if (vertices is not None) and (len(vertices) == n):
    vv = vertices
  else:
    vv = range(1, n+1)    # numbers 1..n
  # printing
  for i in range(n):
    print(vv[i], ": ", sep = "", end="")
    for j in range(n):
      if matrix[i][j] == 1:
        print(vv[j], end=" ")
    print()

def print_dict(graph):
  """
  Prints a graph given by the adjacency list (or just a dictionary).
  """
  for v in graph:
    print(v, ": ", sep = "", end="")
    for u in graph[v]:
      print(u, end=" ")
    print()

## Class *Graph*

In [72]:
class Graph:
    def __init__(self, graph=None):
        if graph is None:
            graph = {}
        self.graph = graph

    # dict initializer
    @classmethod
    def from_dict(cls, graph):
        return cls(graph)

    # array initializer
    @classmethod
    def from_matrix(cls, matrix, vertices = None):
        if (vertices is None) or (len(vertices) != len(matrix)):
            vertices = [*range(1, len(matrix) + 1)]
        return cls.from_dict(cls._matrix_to_dict(matrix, vertices))

    # two private methods matrix <-> dictionaries
    def _matrix_to_dict(matrix, vertices: list) -> dict:
        """
        Converts a graph given as an adjacency matrix to a graph in dict form.
        """
        n = len(matrix)   # the order of a graph
        # take the proper names of vertices
        if (vertices is not None) and (len(vertices) == n):
          vv = vertices
        else:
          vv = range(1, n+1)    # numbers 1..n

        res_dict = {}
        for i, v in enumerate(vv):
            neighbours = [vv[j] for j, edge in enumerate(matrix[i]) if edge]
            res_dict[v] = neighbours
        return res_dict

    def _dict_to_matrix(self, _dict: dict) -> np.array:
        """
        Converts a graph in dict form to its adjacency matrix.
        """
        n = len(_dict)
        vertices = [*_dict.keys()]
        matrix = np.zeros(shape = (n, n), dtype=int)
        for i in range(n):
          for neighbor in _dict[vertices[i]]:
            j = vertices.index(neighbor)
            matrix[i][j] = 1
        return matrix

    def vertices(self) -> list:
        """
        Returns list of vertices of the graph.
        """
        return [*self.graph.keys()]

    def matrix(self) -> np.array:
        """
        Returns the adjacency matrix of the graph.
        """
        return self._dict_to_matrix(self.graph)

    # redefinition of print for objects of class Graph
    def __str__(self):
        res = ""
        for v in self.graph:
            res += f"{v}:"
            for u in self.graph[v]:
                res += f" {u}"
            res += "\n"
        return res

    # The following is for free thanks to the above
    def to_neighbourlist(self, filename: str):
        """
        Saves a graphs to a text file as a neighbour dict.
        Filename is a file path.
        """
        file = open(filename, "w")  # open textfile for writing
        file.write(str(self))
        file.close()

    # Modyfying graphs
    def add_vertex(self, vertex):
        """
        Adds a new vertex to the graph.
        """
        if vertex not in self.graph:
            self.graph[vertex] = []

    def del_vertex(self, vertex):
        """
        Removes a vertex from the graph.
        """
        if vertex in self.graph:
            self.graph.pop(vertex)
            for u in self.graph:
                if vertex in self.graph[u]:
                    self.graph[u].remove(vertex)

    def add_arc(self, arc):
        """
        Given pair of vertices (arc variable) add an arc to the graph
        We consider simple, directed graphs.
        """
        u, v = arc
        self.add_vertex(u)
        self.add_vertex(v)
        if v not in self.graph[u]:
            self.graph[u].append(v)

    def add_edge(self, edge: list):
        """
        Given pair of vertices (edge variable) add an edge to existing graph.
        We consider simple, undirected graphs, as symmetric digraphs without loops.
        """
        u, v = edge
        if u == v:
            raise ValueError("Loops are not allowed!")
        self.add_vertex(u)
        self.add_vertex(v)
        if v not in self.graph[u]:
            self.graph[u].append(v)
        if u not in self.graph[v]:
            self.graph[v].append(u)

    # reading from a file
    @staticmethod
    def from_edges(filename: str, directed = 0):
        """
        Read the graph from file, that in each line contains either
        the description of a vertex (one word) or
        the description of an edge/arc (at least 2 words).
        The resulting graph is returned as a neighbourhood list.
        Variable "filename" contains the whole path to the file.
        """
        graph = Graph()
        file = open(filename, "r")          # open the file to read
        for line in file:                   # for each line of the file
          words = line.strip().split()      # splits the line into words
          if len(words) == 1:               # one word - vertex description
            graph.add_vertex(words[0])
          elif len(words) >= 2:             # at least two words, first two are the edge description
            if directed:
              graph.add_arc([words[0], words[1]])
            else:
              graph.add_edge([words[0], words[1]])
        file.close()
        return graph

    @staticmethod
    def random_graph(n: int, p: float):
        """
        Creates a random graph in G(n, p) model.
        """
        rand_graph = Graph()
        for i in range(1, n + 1):
            rand_graph.add_vertex(i)
            for j in range(1, i):
                if random() < p:
                    rand_graph.add_edge([i, j])
        return rand_graph

    @staticmethod
    def cycle(n: int):
        """
        Creates a cycle C_n on n vertices
        """
        cycle = Graph()
        for i in range(1,n):
          cycle.add_edge((i, i+1))
        cycle.add_edge((1, n))
        return cycle

    def __len__(self):
        return len(self.graph)

    def __contains__(self, vertex):
        return vertex in self.graph

    def __getitem__(self, vertex):
        return self.graph[vertex]

    def __eq__(self, graph):
        return self.graph == graph.graph

    def __ne__(self, graph):
        return self.graph != graph.graph

    def _prufer(self):
        """
        Private: returns the Prufer code of the graph.
        It is neccessary the graph to be a tree (it is not checked).
        """
        code = ""
        for i in range(len(self) - 2):
          for x in sorted(self.graph):
            if len(self.graph[x]) == 1:   # the least leaf
              break
          code += str(self.graph[x][0]) + " "
          self.del_vertex(x)
        return code

    def Prufer(self):
        """
        Returns the Prufer code of the graph.
        It is neccessary the graph to be a tree (it is not checked).
        """
        return deepcopy(self)._prufer()

    def Prufer2(self):
        """
        Returns the Prufer code of the graph.
        It is neccessary the graph to be a tree (it is not checked).
        """
        code = ""
        tr = deepcopy(self.graph)
        for i in range(len(self) - 2):
          for x in sorted(tr):
            if len(tr[x]) == 1:   # the least leaf
              break
          v = tr[x][0] # the neighbour of ther least leaf
          code += str(v) + " "
          tr[v].remove(x)
          tr.pop(x)
        return code

    @staticmethod
    def tree_from_Prufer(code: str):
        """
        Creates a tree from a Prufer code.
        """
        tree = Graph()
        clist = [int(x) for x in code.strip().split()]
        n = len(clist) + 2
        vert = [x for x in range(1, n+1)]
        for v in vert:
          tree.add_vertex(v)
        for i in range(n-2):
          for x in vert:
            if x not in clist: # least leaf
              break
          v = clist.pop(0)
          tree.add_edge([x, v])
          vert.remove(x)
        tree.add_edge(vert)
        return tree

## Use of code

## Part 3 (Prüfer codes)

In [25]:
%%writefile tree1.text
1 3
2 5
3 5
4 5
5 6

Writing tree1.text


In [73]:
tree1 = Graph.from_edges("tree1.text")
print(tree1)

1: 3
3: 1 5
2: 5
5: 2 3 4 6
4: 5
6: 5



In [74]:
print(tree1.Prufer2())

3 5 5 5 


In [75]:
print(tree1)

1: 3
3: 1 5
2: 5
5: 2 3 4 6
4: 5
6: 5



In [76]:
print(tree1.Prufer())

3 5 5 5 


In [77]:
print(tree1)

1: 3
3: 1 5
2: 5
5: 2 3 4 6
4: 5
6: 5



In [78]:
tree4 = Graph.tree_from_Prufer("4 4 4 4")
print(tree4)

1: 4
2: 4
3: 4
4: 1 2 3 5 6
5: 4
6: 4



In [79]:
print(tree4.Prufer())

4 4 4 4 


In [80]:
tree2 = Graph.tree_from_Prufer("2 2 2 2")
print(tree2)

1: 2
2: 1 3 4 5 6
3: 2
4: 2
5: 2
6: 2



In [81]:
tree3 = Graph.tree_from_Prufer("4 3 3 2 1")
print(tree3)

1: 2 7
2: 3 1
3: 4 6 2
4: 5 3
5: 4
6: 3
7: 1



### Part2

In [15]:
%%writefile edges.txt
a b
b c
b d
d
d c
e
f


Writing edges.txt


In [16]:
%cat edges.txt

a b
b c
b d
d
d c
e
f


In [17]:
graph2 = Graph.from_edges("edges.txt")
print(graph2)

a: b
b: a c d
c: b d
d: b c
e:
f:



In [18]:
graph2.to_neighbourlist("neighbourhood.txt")

In [19]:
%cat "neighbourhood.txt"

a: b
b: a c d
c: b d
d: b c
e:
f:


In [20]:
!wget https://raw.githubusercontent.com/pgordin/GraphsSN_2026_1/refs/heads/main/weighted0.txt

--2026-11-05 11:51:08--  https://raw.githubusercontent.com/pgordin/GraphsSN_2026_1/refs/heads/main/weighted0.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 114 [text/plain]
Saving to: ‘weighted0.txt’

weighted0.txt       100%[===================>]     114  --.-KB/s    in 0s      

2026-11-05 11:51:08 (1.90 MB/s) - ‘weighted0.txt’ saved [114/114]



In [21]:
graph3 = Graph.from_edges("weighted0.txt")
print(graph3)

A: B E
B: A C D
E: A D F H
C: B D F G
D: B C E F
F: C D E G H
G: C F H I
H: E F G
I: G



In [22]:
graph4 = Graph.random_graph(10, 1/3)
print(graph4)

1: 2 10
2: 1 5 8 10
3: 4 7 9
4: 3
5: 2
6: 8 9
7: 3 9
8: 2 6 9
9: 3 6 7 8
10: 1 2



In [23]:
print(Graph.cycle(6))

1: 2 6
2: 1 3
3: 2 4
4: 3 5
5: 4 6
6: 5 1



### Part1

In [9]:
vertices = ["a", "b", "c", "d"]
matrix = np.array([[0,1,0,0],[1,0,1,0],[0,1,0,1],[0,0,0,1]])
print(vertices)
print(matrix)
print("---------------------------")
print_matrix(vertices, matrix)
print("---------------------------")
print_matrix(None,matrix)

['a', 'b', 'c', 'd']
[[0 1 0 0]
 [1 0 1 0]
 [0 1 0 1]
 [0 0 0 1]]
---------------------------
a: b 
b: a c 
c: b d 
d: d 
---------------------------
1: 2 
2: 1 3 
3: 2 4 
4: 4 


In [3]:
graph_dict = {
  "a": ["b"],
  "b": ["a", "c"],
  "c": ["b", "d"],
  "d": ["c"]
}
print(graph_dict)
print("---------------------------")
print_dict(graph_dict)

{'a': ['b'], 'b': ['a', 'c'], 'c': ['b', 'd'], 'd': ['c']}
---------------------------
a: b 
b: a c 
c: b d 
d: c 


In [6]:
graph = Graph(graph_dict)
print(graph)

a: b
b: a c
c: b d
d: c



In [7]:
graph2 = Graph.from_dict(graph_dict)
print(graph2)

a: b
b: a c
c: b d
d: c



In [10]:
graph1 = Graph.from_matrix(matrix, vertices)
print(graph1)

a: b
b: a c
c: b d
d: d



In [11]:
print(Graph(graph_dict))  # the same result

a: b
b: a c
c: b d
d: c



In [ ]:
print(graph1.vertices())

['a', 'b', 'c', 'd']


In [ ]:
print(graph1.matrix())

[[0 1 0 0]
 [1 0 1 0]
 [0 1 0 1]
 [0 0 0 1]]


In [ ]:
graph1.add_vertex("e")
print(graph1)

a: b
b: a c
c: b d
d: d
e:



In [ ]:
graph1.add_edge(["e", "f"])
print(graph1)

a: b
b: a c
c: b d
d: d
e: f
f: e



In [ ]:
graph1.add_arc(["e", "a"])  # breaking the symmetry
print(graph1)

a: b
b: a c
c: b d
d: d
e: f a
f: e



In [ ]:
graph1.add_edge(["e", "a"])  # restoring the symmetry
print(graph1)

a: b e
b: a c
c: b d
d: d
e: f a
f: e



In [ ]:
graph1.add_edge(["e", "f"]) # do nothing, an edge already exists
print(graph1)

a: b e
b: a c
c: b d
d: d
e: f a
f: e



In [ ]:
graph1.del_vertex("f")  # removing a vertex
print(graph1)

a: b e
b: a c
c: b d
d: d
e: a



In [ ]:
graph1.add_edge(["e", "e"]) # an error

ValueError: Loops are not allowed!

In [ ]:
graph1.add_arc(["e", "e"]) # OK - loops are allowed in digraphs
print(graph1)

a: b e
b: a c
c: b d
d: d
e: a e



In [ ]:
print(Graph.cycle(6))

1: 2 6
2: 1 3
3: 2 4
4: 3 5
5: 4 6
6: 5 1

